# Day 3, Lesson 3.1: Function calling (how an LLM uses a tool)

## Step 1: Setup

In [1]:
import json
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-3.5-flash-lite"
print("Ready:", MODEL)

Ready: gemini-3.5-flash-lite


## Step 2: Two toy Python functions

In [2]:
def get_fake_weather(city: str) -> dict:
    return {"city": city, "temperature_c": 31.2, "humidity_pct": 70, "condition": "partly cloudy",
            "note": "fake demo data"}

def celsius_to_fahrenheit(celsius: float) -> dict:
    return {"celsius": celsius, "fahrenheit": round(celsius * 9 / 5 + 32, 1)}

print(get_fake_weather("Mumbai"))
print(celsius_to_fahrenheit(31.2))

{'city': 'Mumbai', 'temperature_c': 31.2, 'humidity_pct': 70, 'condition': 'partly cloudy', 'note': 'fake demo data'}
{'celsius': 31.2, 'fahrenheit': 88.2}


## Step 3: Describe the tools to Gemini (JSON schema)

In [3]:
TOOL_DECLARATIONS = [
    types.FunctionDeclaration(
        name="get_fake_weather",
        description="Get the current weather (temperature, humidity, condition) for a city.",
        parameters_json_schema={
            "type": "object",
            "properties": {"city": {"type": "string", "description": "City name, e.g. Mumbai"}},
            "required": ["city"],
        },
    ),
    types.FunctionDeclaration(
        name="celsius_to_fahrenheit",
        description="Convert a temperature from degrees Celsius to degrees Fahrenheit.",
        parameters_json_schema={
            "type": "object",
            "properties": {"celsius": {"type": "number", "description": "Temperature in °C"}},
            "required": ["celsius"],
        },
    ),
]

TOOLS = {"get_fake_weather": get_fake_weather, "celsius_to_fahrenheit": celsius_to_fahrenheit}

AGENT_RULES = """You are a weather assistant.
- Use a tool only when it is needed to answer the question.
- Give temperatures in °C. Convert to Fahrenheit only if the user asks for Fahrenheit.
- If no tool can answer the question, say so honestly. Do not guess."""

CONFIG = types.GenerateContentConfig(
    system_instruction=AGENT_RULES,
    tools=[types.Tool(function_declarations=TOOL_DECLARATIONS)],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    temperature=0.1,
)
print(json.dumps(TOOL_DECLARATIONS[0].parameters_json_schema, indent=2))

{
  "type": "object",
  "properties": {
    "city": {
      "type": "string",
      "description": "City name, e.g. Mumbai"
    }
  },
  "required": [
    "city"
  ]
}


## Step 4: One round trip, step by step

In [4]:
contents = [types.Content(role="user", parts=[types.Part.from_text(text="How hot is Mumbai right now?")])]

response = client.models.generate_content(model=MODEL, contents=contents, config=CONFIG)
call = response.function_calls[0]
print("Gemini wants to call:", call.name, "with", dict(call.args))
print("Text answer so far:", response.text)

Gemini wants to call: get_fake_weather with {'city': 'Mumbai'}
Text answer so far: None


In [5]:
result = TOOLS[call.name](**call.args)
print("Tool result:", result)

contents.append(response.candidates[0].content)
contents.append(types.Content(role="user", parts=[
    types.Part.from_function_response(name=call.name, response=result)
]))

final = client.models.generate_content(model=MODEL, contents=contents, config=CONFIG)
if final.function_calls:
    print("Gemini asked for another tool:", [(c.name, dict(c.args)) for c in final.function_calls])
    print("-> that's why we need a LOOP (Step 5)")
else:
    print("Final answer:", final.text)
print("\nThe conversation now has", len(contents) + 1, "messages:",
      [c.role for c in contents] + ["model"])

Tool result: {'city': 'Mumbai', 'temperature_c': 31.2, 'humidity_pct': 70, 'condition': 'partly cloudy', 'note': 'fake demo data'}


Final answer: Mumbai is currently 31.2°C and partly cloudy, with a humidity of 70%.

The conversation now has 4 messages: ['user', 'model', 'user', 'model']


## Step 5: A loop that handles any number of tool calls

In [6]:
import time

from google.genai import errors

def generate(contents, retries=4):
    for attempt in range(1, retries + 1):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=CONFIG)
        except errors.APIError as error:
            if error.code in (429, 503) and attempt < retries:
                wait = 30 if error.code == 429 else 5 * attempt
                print(f"   (Gemini {error.code}: waiting {wait}s, then retrying)")
                time.sleep(wait)
                continue
            raise

def run_with_tools(question, max_steps=5, verbose=True):
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=question)])]
    tools_used = []

    for step in range(1, max_steps + 1):
        response = generate(contents)
        calls = response.function_calls or []
        if not calls:
            return {"answer": response.text, "tools_used": tools_used}

        contents.append(response.candidates[0].content)
        result_parts = []
        for call in calls:
            args = dict(call.args or {})
            try:
                result = TOOLS[call.name](**args)
            except KeyError:
                result = {"error": f"unknown tool {call.name}"}
            except TypeError as error:
                result = {"error": f"bad arguments: {error}"}
            tools_used.append(call.name)
            if verbose:
                print(f"   step {step}: {call.name}({args}) -> {result}")
            result_parts.append(types.Part.from_function_response(name=call.name, response=result))
        contents.append(types.Content(role="user", parts=result_parts))

    return {"answer": "Sorry, I could not finish this question.", "tools_used": tools_used}

## Step 6: Test: one tool, two tools, no tool

In [7]:
for question in [
    "How hot is Mumbai right now?",
    "What is 40 degrees Celsius in Fahrenheit?",
    "How hot is Mumbai right now, in Fahrenheit?",
    "Hi! What can you help me with?",
    "What will the weather be in Mumbai next week?",
]:
    print(f"Q: {question}")
    result = run_with_tools(question)
    print(f"   tools used: {result['tools_used'] or 'none'}")
    print(f"   answer: {result['answer'].strip()[:200]}\n")

Q: How hot is Mumbai right now?


   step 1: get_fake_weather({'city': 'Mumbai'}) -> {'city': 'Mumbai', 'temperature_c': 31.2, 'humidity_pct': 70, 'condition': 'partly cloudy', 'note': 'fake demo data'}


   tools used: ['get_fake_weather']
   answer: The temperature in Mumbai right now is 31.2°C, and it is partly cloudy with a humidity of 70%.

Q: What is 40 degrees Celsius in Fahrenheit?


   step 1: celsius_to_fahrenheit({'celsius': 40}) -> {'celsius': 40, 'fahrenheit': 104.0}


   tools used: ['celsius_to_fahrenheit']
   answer: 40 degrees Celsius is equal to 104 degrees Fahrenheit.

Q: How hot is Mumbai right now, in Fahrenheit?


...

   step 2: celsius_to_fahrenheit({'celsius': 31.2}) -> {'celsius': 31.2, 'fahrenheit': 88.2}


   (Gemini 429: waiting 30s, then retrying)


   tools used: ['get_fake_weather', 'celsius_to_fahrenheit']
   answer: Mumbai is currently 31.2°C (88.2°F) and partly cloudy.

Q: Hi! What can you help me with?


   tools used: none
   answer: Hi! I can help you check the current weather conditions (like temperature and humidity) for different cities. If you'd like, I can also convert temperatures from Celsius to Fahrenheit for you. Just le

Q: What will the weather be in Mumbai next week?


   tools used: none
   answer: I am sorry, but I do not have access to forecasts for next week. I can only check the current weather conditions.

